# ⚙️ Agentic AI for Science (AAI4Science) Hackathon 2025



This notebook demonstrates the workflow for using the AtomGPT (https://atomgpt.org) API and
agentic AI functionalities to create, test, and run simple agentic tasks in the context of
the AAI4Science (Agentic AI for Science) Hackathon 2025.

The example shows:
1. How to install and configure dependencies.
2. How to initialize AGAPI and OpenAI-compatible clients.
3. How to perform simple API-based interactions.
4. How to define and run function tools and asynchronous agents.

Author: Dr. Kamal Choudhary (kchoudh2@jhu.edu)

Reference: https://doi.org/10.1007/s40192-025-00410-9

Event: https://www.eventbrite.com/e/agentic-ai-for-science-aai4science-hackathon-2025-tickets-1797906650189






Installs the required Python packages:
- `openai-agents`: Provides Agentic AI abstraction tools (Agent, Runner, function_tool).
- `agapi`: AtomGPT API client for connecting to the AtomGPT.org endpoint.

This ensures all modules required for subsequent agentic operations are available.

In [ ]:
!uv pip install openai-agents agapi

Using Python 3.12.12 environment at: /usr
Resolved 42 packages in 674ms
Prepared 8 packages in 380ms
Uninstalled 3 packages in 105ms
Installed 8 packages in 54ms
 + agapi==2025.9.15
 + colorama==0.4.6
 + griffe==1.14.0
 - openai==1.109.1
 + openai==2.7.1
 + openai-agents==0.5.0
 - pydantic==2.11.10
 + pydantic==2.12.4
 - pydantic-core==2.33.2
 + pydantic-core==2.41.5
 + types-requests==2.32.4.20250913


Instructions:

1. Visit https://atomgpt.org/
2. Navigate to: Profile → Settings → Account → API Keys
3. Create or view your API key (looks like 'sk-xxxxxxxxx').
4. Paste the key below in the variable `api_key`.

⚠️ Note: For security, do not share or hardcode your real API key in public repositories.

In [ ]:
api_key=API_KEY

Demonstrates using the AGAPI client to query the AtomGPT API directly.

Steps:
1. Initialize the `Agapi` client with the provided API key.
2. Send a simple query ("What’s the capital of US") to test the connection.
3. Print the response returned by the AtomGPT system.

Expected Output:
"The capital of US is Washington, D.C."

In [ ]:
from agapi.client import Agapi
client = Agapi(api_key=api_key)
r = client.ask("Whats the capital of US")
print(r)


The capital of the United States is **Washington, D.C.**


# ⚙️ Agentic AI with Function Tool Example


This section introduces the concept of an agentic workflow using OpenAI-compatible Agents.

Modules used:
- `AsyncOpenAI`: Async API client for concurrent operations.
- `function_tool`: Decorator for defining callable tools.
- `Agent`, `Runner`, `OpenAIChatCompletionsModel`: Core classes for defining, configuring, and executing AI agents.
- `set_tracing_disabled`: Disables tracing for cleaner execution during demos.

Key Steps:
1. Define an asynchronous OpenAI client using AtomGPT API.
2. Create a function tool (`get_weather`) that simulates retrieving weather data.
3. Define an agent with instructions, model, and tool integration.
4. Run the agent asynchronously using the `Runner.run()` method.

Expected Behavior:
The agent uses the tool automatically when the user asks for weather, returning a formatted response.


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://atomgpt.org/api",
    api_key=api_key
)

result = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Whats the capital of US?"}
    ],
    reasoning_effort="high"
)

print(result.choices[0].message.content)



The capital of the United States is Washington, D.C.


In [ ]:
import requests
from openai import AsyncOpenAI
from agents import function_tool, Agent, OpenAIChatCompletionsModel
from agents import set_tracing_disabled, Runner, ModelSettings

set_tracing_disabled(disabled=True)

client = AsyncOpenAI(
    base_url="https://atomgpt.org/api",
    api_key=api_key
)

# -----------------------------
# 🌤️ Function Tool Definition
# -----------------------------
"""
Defines a callable function tool that the AI agent can use to retrieve weather information.

Parameters:
- city (str): The name of the city for which weather is requested.

Returns:
- str: A formatted string describing the current weather conditions.

The decorator `@function_tool` registers the function so that the agent can decide to call it automatically
when the query requires it (e.g., “What’s the weather in New York City?”).
"""

def ask_for_weather(location: str, api_key: str):
    url = f"https://atomgpt.org/weather?location={location}&APIKEY={api_key}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

def weather_json_to_text(weather_json: dict) -> str:
  return (
      f"In {weather_json['location']}, {weather_json['country']}, "
      f"the weather is currently {weather_json['weather']} with a temperature of "
      f"{weather_json['temperature']}°{weather_json['units']}. It feels like "
      f"{weather_json['feels_like']}°, humidity is {weather_json['humidity']}%, "
      f"wind speed is {weather_json['wind_speed']} mph, and the pressure is "
      f"{weather_json['pressure']} hPa."
      "Be carefeel with numerical data."
  )

@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    result = weather_json_to_text(ask_for_weather(city, api_key = api_key))
    return result + " Shorten your answer."

    # print(f"[debug] getting weather for {city}")
    # return f"The weather in {city} is sunny. Temperature: 62°F. Humidity: 45%. Answer in a single sentense."


# -----------------------------
# 🧠 Agent Initialization
# -----------------------------
"""
Creates an Agent named “Assistant” with custom behavior and attached tools.

Parameters:
- name (str): Agent’s name for identification.
- instructions (str): Contextual behavior instructions for the model.
- model (OpenAIChatCompletionsModel): Backend model for text generation.
- tools (list): List of callable tools available to the agent (e.g., get_weather).

Optional:
ModelSettings can be used to control tool invocation mode, reasoning depth, etc.
"""

agent = Agent(
    name="Assistant",
    instructions="You're a helpful assistant. You respond in a format that is useful for Enterprise Executives.",
    model=OpenAIChatCompletionsModel(
        model="openai/gpt-oss-20b",
        openai_client=client
    ),
    # model_settings=ModelSettings(
    #     tool_choice="auto",
    # ),
    tools=[get_weather],
)

# -----------------------------
# 🚀 Run the Agent
# -----------------------------
"""
Runs the agent asynchronously using the Runner utility.

Query:
- "What's the weather in New York City?"

Expected Flow:
1. The model identifies that the `get_weather` tool can be used.
2. The tool executes, returning the weather string.
3. The final output is printed as the agent’s response.

Expected Output:
"The weather in New York City is sunny. Temperature: 62°F. Humidity: 45%."
"""

result = await Runner.run(agent, "What's the weather in New York City?")
print(result.final_output)

**New York City (Current)**  
- **Condition:** Overcast  
- **Temperature:** 53 °F (≈12 °C)  
- **Wind:** 22 mph (≈36 km/h)  
- **Humidity:** 37 %  
- **Pressure:** 1 018 hPa  

*All values are current; update hourly.*


In [ ]:
result = await Runner.run(agent, "What's the weather in Moscow?")
print(result.final_output)

**Moscow Weather Snapshot (as of [UTC date & time])**

| Metric | Value |
|--------|-------|
| **Condition** | Light rain |
| **Temperature** | **47.7 °F** (≈ 8.4 °C) |
| **Feels‑Like** | 42.9 °F (≈ 6.2 °C) |
| **Humidity** | 73 % |
| **Wind** | 10.6 mph (≈ 17 km/h) |
| **Pressure** | 1018 hPa |

**Key Implications for Executives**

- **Travel & Mobility:** Light rain may affect ground congestion, particularly in central Moscow. Consider potential delays of 10–20 min for transit‑based travel or flights from Sheremetyevo/Domodedovo.
- **Operational Impact:** If your organization operates field sites in Moscow, schedule outdoor activities for periods after 2 pm to avoid the heaviest rain.
- **Safety & Comfort:** High humidity combined with wind may feel cooler; advise staff to carry umbrellas and wear appropriate jackets.
- **Event Planning:** If you have scheduled meetings or events, ensure reliable indoor backup and real‑time weather updates for last‑minute adjustments.

*Let me know i

# Task 1: Make a tool calling to get current weather in Baltimore modifying the scipt/function above and using the function such as https://atomgpt.org/weather?location=Baltimore&APIKEY=sk-XYZ

In [ ]:
def ask_for_weather(location: str, api_key: str):
    url = f"https://atomgpt.org/weather?location={location}&APIKEY={api_key}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

def weather_json_to_text(weather_json: dict) -> str:
  return (
      f"In {weather_json['location']}, {weather_json['country']}, "
      f"the weather is currently {weather_json['weather']} with a temperature of "
      f"{weather_json['temperature']}°{weather_json['units']}. It feels like "
      f"{weather_json['feels_like']}°, humidity is {weather_json['humidity']}%, "
      f"wind speed is {weather_json['wind_speed']} mph, and the pressure is "
      f"{weather_json['pressure']} hPa."
      "Be carefeel with numerical data."
  )

@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    result = weather_json_to_text(ask_for_weather(city, api_key = api_key))
    return result + " Shorten your answer."

# Task 2: Make a tool calling to get total number of materials in the JARVIS-DFT database using the function such as https://atomgpt.org/jarvis_dft/query?elements="Si,C"&APIKEY=sk-XYZ

In [ ]:
from typing import List

def ask_for_db(elements : List[str], api_key: str) -> str:
    elements_str = ""
    for element_idx, element_str in enumerate(elements):
      elements_str += f"{element_str}"
      if element_idx < len(elements) - 1:
        elements_str += ","
    elements_str = '"' + elements_str + '"'
    url = f'https://atomgpt.org/jarvis_dft/query?elements={elements_str}&APIKEY={api_key}'
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

def count_materials_in_db(elements : List[str]) -> int:
  "Get the total number of materials in the JARVIS-DFT database with elements from elements list: for example ['Si','Ag','Li',]."
  total_n = ask_for_db(elements, api_key = api_key)["total"]
  assert isinstance(total_n,int)
  return total_n

@function_tool
def count_materials_in_db_tool(elements : List[str]) -> int:
  return f"Total number of materials is {count_materials_in_db(elements)} . Your answer should start with 'There are {count_materials_in_db(elements)} ...'."

In [ ]:
client = AsyncOpenAI(
    base_url="https://atomgpt.org/api",
    api_key=api_key
)

agent = Agent(
    name="Assistant",
    instructions="You're a helpful assistant. You should respond in a clear way, not to short but not to long. You are an expert in chemistry that is willing to help people with your knowledge.",
    model=OpenAIChatCompletionsModel(
        model="openai/gpt-oss-20b",
        openai_client=client
    ),
    tools=[count_materials_in_db_tool],
)

result = await Runner.run(agent, "How many materials with Na and Ca are there in JARVIS-DFT database?")
print(result.final_output)

There are 6 materials with Na and Ca in the JARVIS‑DFT database.


# Task 3: Make a tool calling to latest 10 papers on chemical compound MgB2 from arXiv repository using the function such as https://atomgpt.org/arxiv?query=MgB2&APIKEY=sk-XYZ

In [ ]:
import datetime

sync_client = OpenAI(
    base_url="https://atomgpt.org/api",
    api_key=api_key
)

def ask_for_papers(chemical_compound : str, api_key: str) -> str:
    url = f'https://atomgpt.org/arxiv?query={chemical_compound}&APIKEY={api_key}'
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

def extract_between_arrows(text: str) -> str | None:
    start = text.find("<<<")
    end = text.find(">>>", start + 3)
    if start != -1 and end != -1:
        return text[start + 3:end].strip()
    return None

def ask_binary_question(question : str):
  prompt = f"Your a text analizing agent. All the data for analysis will be in the question prompt. You should answer the question in a single word YES or NO in a <<<...>>> brakets as <<<No>>> or <<<Yes>>>."
  answer = sync_client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": question},
            ],
            temperature=0.2,
  ).choices[0].message.content
  key = extract_between_arrows(answer)
  return key

@function_tool
def get_top_n_relevant_papers(compound : str, n : int):
  "Get top n recent papers about the compound."
  papers_json = ask_for_papers(compound, api_key=api_key)
  papers = papers_json["results"]
  papers_sorted = sorted(
    papers,
    key=lambda p: datetime.datetime.fromisoformat(p["published"]),
    reverse=True
  )
  resulting_papers = []
  for paper in papers_sorted:
    print(f"Looking at paper {paper["title"]}")
    summary = paper["summary"]
    question = f"Is the scientific paper with summary '{summary}' relevant to the search of the papers about the {compound} compound/element?"
    if ask_binary_question(question) == "Yes":
      resulting_papers.append(paper)
      if len(resulting_papers) == n:
        break
  return str(resulting_papers)


In [ ]:
client = AsyncOpenAI(
    base_url="https://atomgpt.org/api",
    api_key=api_key
)

agent = Agent(
    name="Assistant",
    instructions="You're a helpful assistant. You should respond in a clear way, not to short but not to long. You are an expert in chemistry that is willing to help people with your knowledge.",
    model=OpenAIChatCompletionsModel(
        model="openai/gpt-oss-20b",
        openai_client=client
    ),
    tools=[get_top_n_relevant_papers],
)

result = await Runner.run(agent, "Search for resent 5 papers about NaCl compound. Tell me only their urls and titles.")
print(result.final_output)

- **[Tracking Microhydration of the NaCl Rocksalt Molecule in Helium Nanodroplets by Penning Ionization Electron Spectroscopy](http://arxiv.org/abs/2510.22000v1)**  
- **[Accessing Energetically Restricted Optical Transitions in a Single Free‑Base Porphyrin Molecule](http://arxiv.org/abs/2511.00630v1)**  
- **[Observation of counterion binding in the inner Helmholtz layer at the ionic surfactant‑water interface](http://arxiv.org/abs/2510.19554v1)**  
- **[Microfluidic Study of Evaporation‑Driven Crystallization of Saline and Ammonia Brines under Hydrogen Flow](http://arxiv.org/abs/2510.20321v1)**  
- **[Study of the Molecular Level Mechanism of Nanoscale Alternating Current Electrohydrodynamic Flow](http://arxiv.org/abs/2510.21754v1)**


# Task 4: There are exactly three positive real numbers $k$ such that the function
$$f(x) = \frac{(x - 18)(x - 72)(x - 98)(x - k)}{x}$$
defined over the positive real numbers achieves its minimum value at exactly two positive real numbers $x$. Find the sum of these three values of $k$. (Using any chatbot such as chatgpt.com, claude.ai etc. that you like. Correct answer: 240

# Task 5: Alex divides a disk into four quadrants with two perpendicular diameters intersecting at the center of the disk. He draws 25 more line segments through the disk, drawing each segment by selecting two points at random on the perimeter of the disk in different quadrants and connecting those two points. Find the expected number of regions into which these 27 line segments divide the disk. Correct answer: 204.

# Task 6+: Identify problems where chatbots such as chatgpt.com etc. fail, and suggest their solution with tool calling.

# Submit response to: https://forms.gle/AycYgYj4ZZoBZE7m9